# Scenario 6: Custom Evaluation Pipeline with Galileo

## What You'll Learn

In this notebook, you will learn how to:

1. **Define custom code-based metrics** — write Python functions that score your LLM outputs using your own logic
2. **Use the `@log` decorator** — automatically instrument regular Python functions as Galileo spans (no manual trace management needed)
3. **Log manual traces** — use `GalileoLogger` directly when you need full control over trace structure
4. **Enable safety metrics** — turn on built-in metrics that detect harmful content (toxicity, sexism, prompt injection)
5. **Inspect available scorers** — discover all 140+ built-in metrics available in Galileo
6. **Use distributed tracing** — pass trace context across microservices so they share one end-to-end trace

---

## Why Custom Metrics?

Galileo comes with **140+ built-in metrics** (safety, quality, RAG relevance, etc.), but your application may have **domain-specific requirements** that no generic metric can cover. For example:

- *"Responses must be between 20–200 characters"* (length compliance)
- *"Must mention our product name in every answer"* (brand consistency)
- *"Must not use passive voice"* (style enforcement)
- *"Must include a citation from our knowledge base"* (factual grounding)

Custom metrics let you define this scoring logic in Python. Your function runs alongside the built-in metrics, and the results appear together in the Galileo console.

---

## Key Concepts

| Concept | What It Means |
|---|---|
| **Local Metric** | A Python function you write that scores traces. It runs in your process (not on Galileo's servers) and returns a float between 0.0 and 1.0. |
| **Scorer** | Any metric evaluator — can be a built-in Galileo metric or a custom one you define. |
| **`@log` Decorator** | A decorator that auto-instruments Python functions as Galileo spans. When decorated functions call other decorated functions, the spans nest automatically. |
| **Distributed Tracing** | A pattern for passing trace context across microservices (via HTTP headers) so that spans from different services appear in one unified trace. |

---

## Prerequisites

- A Galileo account and API key (`GALILEO_API_KEY` in `.env`)
- An OpenAI API key (`OPENAI_API_KEY` in `.env`)
- The `galileo` Python SDK installed (`uv sync` from the repo root)

### Load Environment Variables

The cell below loads your `.env` file, which must contain `GALILEO_API_KEY` and `OPENAI_API_KEY`. It searches both the current directory and the parent directory so the notebook works whether you run it from `agents/` or the repo root.

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


### Imports and Configuration

Key imports explained:

- **`GalileoLogger`** — the core logger for manually creating traces and spans
- **`galileo_context`** — manages the global Galileo session (project, log stream, flush)
- **`galileo_log`** (the `@log` decorator) — auto-instruments Python functions as Galileo spans, so you don't need manual `start_trace`/`conclude` calls
- **`LocalMetricConfig`** — wraps your custom Python scoring function so Galileo can register and display it as a metric
- **`enable_metrics`** — activates metrics (built-in or custom) on a log stream
- **`Scorers`** — lists all available metric scorers in your Galileo environment
- **`Message` / `MessageRole`** — data classes for constructing chat messages when logging LLM spans manually

In [ ]:
from galileo import GalileoLogger, GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.decorator import log as galileo_log
from galileo.log_streams import LocalMetricConfig, create_log_stream, enable_metrics, get_log_stream
from galileo.projects import delete_project
from galileo.scorers import Scorers

PROJECT_NAME = 'GalileoEval_S6_CustomEval'
LOG_STREAM_NAME = 'custom-eval-pipeline'
MODEL = 'gpt-4o-mini'

## 1. Initialize Galileo

Before logging anything, we need to initialize a Galileo session by specifying a **project** and a **log stream**. Think of a project as a folder and a log stream as a subfolder where all traces for a particular experiment or environment are stored. The code below creates them if they don't already exist, then prints links to the Galileo console where you can view results.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 2. Register a Local Metric

A **local metric** is a Python function you write that scores each trace. Here's how it works:

1. **Your function receives a `trace` object** with properties like `trace.output` (the LLM's response text), `trace.input` (the user's question), and `trace.spans` (the individual steps).
2. **Your function returns a float between 0.0 and 1.0**, where 1.0 means "perfect" and 0.0 means "worst possible."
3. **You wrap it in `LocalMetricConfig`** to give it a display name that Galileo can show in the console.
4. **You call `enable_metrics()`** to activate it on your log stream. From that point, every trace logged to this stream will be scored by your function.

**This example: `response_length_scorer`**
- Returns **1.0** if the response is between 20–200 characters (the "sweet spot")
- Returns a **proportionally lower score** if the response is too short (e.g., 10 chars → 0.5)
- Returns a **gradually decreasing score** if the response is too long (penalizes verbosity)

The metric runs **locally in your Python process** — your scoring code never leaves your machine. Only the resulting score is sent to Galileo for display.

In [5]:
def response_length_scorer(trace, **kwargs) -> float:
    output = str(trace.output or '')
    length = len(output)
    if 20 <= length <= 200:
        return 1.0
    if length < 20:
        return length / 20.0
    return max(0.0, 1.0 - (length - 200) / 500.0)

enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[LocalMetricConfig(name='response_length_quality', scorer_fn=response_length_scorer)],
)


[LocalMetricConfig(name='response_length_quality', scorer_fn=<function response_length_scorer at 0x1100c9b20>, aggregator_fn=None, scorable_types=[<StepType.llm: 'llm'>], aggregatable_types=[<StepType.trace: 'trace'>])]

## 3. Log a Decorated Workflow

The **`@galileo_log` decorator** is the easiest way to instrument your code. You add it to regular Python functions, and Galileo automatically creates spans from their inputs and outputs — no manual `start_trace` / `conclude` calls needed.

**How it works:**
- Each decorated function becomes a **span** in the trace
- The `span_type` parameter tells Galileo what kind of step it is: `'retriever'`, `'tool'`, `'workflow'`, `'agent'`, or `'llm'`
- When a decorated function **calls another decorated function**, the spans **nest automatically** (child spans appear inside the parent)

**What this example creates:**

```
workflow: qa_pipeline
  ├── retriever: fetch_docs
  └── tool: format_response
```

This is much simpler than the manual approach (Step 4). Use `@galileo_log` when your code is structured as regular Python functions. Use manual logging when you need fine-grained control (e.g., setting token counts or constructing Message objects).

In [6]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

@galileo_log(span_type='retriever')
def fetch_docs(query: str) -> list[str]:
    return [f'Document about: {query}', f'Technical reference for: {query}']

@galileo_log(span_type='tool')
def format_response(docs: list[str], query: str) -> str:
    return f"Based on {len(docs)} documents for {query}: {', '.join(docs)}"

@galileo_log(span_type='workflow')
def qa_pipeline(question: str) -> str:
    docs = fetch_docs(question)
    return format_response(docs, question)

result = qa_pipeline('How does custom evaluation work in Galileo?')
galileo_context.flush()
result


'Based on 2 documents for How does custom evaluation work in Galileo?: Document about: How does custom evaluation work in Galileo?, Technical reference for: How does custom evaluation work in Galileo?'

## 4. Log Manual Traces for Evaluation

Sometimes you need **full control** over trace structure — for example, when you want to set exact token counts, construct specific message formats, or log traces from code that isn't structured as simple function calls. In those cases, use `GalileoLogger` directly.

**The pattern is:**

1. `logger.start_trace(input=...)` — begin a new trace with the user's input
2. `logger.add_llm_span(input=..., output=..., model=...)` — add an LLM call as a span inside the trace
3. `logger.conclude(output=...)` — close the trace with the final output
4. `logger.flush()` — send all buffered traces to Galileo

**What this example does:**

We log 3 traces with intentionally different response qualities to see how our `response_length_scorer` from Step 2 scores them:

| Trace | Response | Expected Score |
|---|---|---|
| "What is the capital of France?" | "The capital of France is Paris." (31 chars) | **1.0** — within the 20–200 sweet spot |
| "Explain quantum computing." | "It's complex." (14 chars) | **~0.7** — too short, penalized |
| "What is machine learning?" | "Machine learning is a subset of AI..." (81 chars) | **1.0** — within range |

After flushing, check the Galileo console to see each trace with its custom metric score.

In [7]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

logger.start_trace(input='What is the capital of France?')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='What is the capital of France?')],
    output=Message(role=MessageRole.assistant, content='The capital of France is Paris.'),
    model=MODEL,
    num_input_tokens=8,
    num_output_tokens=7,
)
logger.conclude(output='The capital of France is Paris.')

logger.start_trace(input='Explain quantum computing.')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Explain quantum computing.')],
    output=Message(role=MessageRole.assistant, content="It's complex."),
    model=MODEL,
    num_input_tokens=5,
    num_output_tokens=2,
)
logger.conclude(output="It's complex.")

logger.start_trace(input='What is machine learning?')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='What is machine learning?')],
    output=Message(
        role=MessageRole.assistant,
        content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.',
    ),
    model=MODEL,
    num_input_tokens=6,
    num_output_tokens=25,
)
logger.conclude(output='Machine learning is a subset of AI that enables systems to learn from experience.')
logger.flush()
'Logged 3 traces'


'Logged 3 traces'

## 5. Enable Safety Metrics

Galileo provides **built-in safety metrics** that automatically detect harmful content patterns. Here we enable five of them:

| Metric | What It Detects |
|---|---|
| `prompt_injection` | Attempts to manipulate or "jailbreak" the LLM (e.g., "ignore all previous instructions...") |
| `input_sexist` | Gender-biased or sexist language in the **user's input** |
| `output_sexist` | Gender-biased or sexist language in the **model's response** |
| `input_toxicity` | Offensive, threatening, or harmful content in the **user's input** |
| `output_toxicity` | Offensive, threatening, or harmful content in the **model's response** |

You can **mix built-in and custom metrics** — they all appear together in the Galileo console.

**Important note:** Calling `enable_metrics()` **replaces** the currently active metric set for that log stream. So the custom `response_length_quality` metric from Step 2 will no longer be active after this cell runs. If you want both custom and built-in metrics active simultaneously, include them all in a single `enable_metrics()` call.

In [8]:
enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        GalileoMetrics.prompt_injection,
        GalileoMetrics.input_sexist,
        GalileoMetrics.output_sexist,
        GalileoMetrics.input_toxicity,
        GalileoMetrics.output_toxicity,
    ],
)


[]

## 6. Inspect Available Scorers

`Scorers().list()` returns **all available metric scorers** in your Galileo environment. This is useful for discovering what metrics exist before choosing which to enable.

Currently there are **140+ scorers** available, organized into categories:

- **RAG metrics** — context adherence, completeness, relevance
- **Agentic metrics** — tool call accuracy, action correctness
- **Safety metrics** — toxicity, sexism, prompt injection, PII detection
- **Quality metrics** — coherence, fluency, instruction adherence
- **Expression metrics** — tone, formality, readability
- **Confidence metrics** — uncertainty estimation

The cell below shows how many scorers are available. You can inspect the full list by running `Scorers().list()` without `len()` to see names and descriptions.

In [9]:
len(Scorers().list())


141

## 7. Distributed Tracing

**Distributed tracing** lets you pass trace context across microservices so that spans from different services appear in one unified trace. This is essential for architectures where an API gateway calls a backend, which calls a model service, which calls a retriever — each running in a separate process.

### How It Works

1. **Service A** starts a trace and calls `logger.get_tracing_headers()` to get HTTP headers containing the trace ID
2. **Service A** passes these headers in its HTTP request to **Service B**
3. **Service B** reads the headers and uses the same trace ID when logging its own spans
4. In the Galileo Console, all spans from both services appear in **one unified trace**

### The Headers

| Header | Purpose |
|--------|---------|
| `X-Galileo-Trace-ID` | Unique identifier for the trace — shared across services |
| `X-Galileo-Parent-ID` | The span ID that new spans should be children of |

### When to Use Distributed Tracing

- **Microservice architectures** — API gateway → backend → ML service → retriever
- **Async pipelines** — Producer → queue → consumer (pass headers via message metadata)
- **Multi-language services** — Python service calling a Node.js service (headers are language-agnostic)

The cell below demonstrates creating a trace in "distributed" mode and extracting the headers.

In [11]:
logger = GalileoLogger(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME, mode='distributed')
logger.start_trace(input='Distributed tracing test')
logger.add_llm_span(
    input=[Message(role=MessageRole.user, content='Test')],
    output=Message(role=MessageRole.assistant, content='OK'),
    model=MODEL,
)
headers = logger.get_tracing_headers()
logger.conclude(output='OK')
logger.flush()
headers


{'X-Galileo-Trace-ID': '06001868-f8ec-4ab4-b500-5b0c69b0a496',
 'X-Galileo-Parent-ID': '06001868-f8ec-4ab4-b500-5b0c69b0a496'}

## 8. Span-Level Code Metrics

By default, custom code metrics run at the **trace level** (they receive the entire trace). But sometimes you want to score **individual spans** — for example, scoring only tool calls or only agent decisions.

`LocalMetricConfig` accepts a `scorable_types` parameter that controls which span types the metric applies to:

| `scorable_types` | What Gets Scored |
|-------------------|-----------------|
| `[StepType.llm]` | Only LLM spans (default) |
| `[StepType.tool]` | Only tool spans |
| `[StepType.agent]` | Only agent spans |
| `[StepType.llm, StepType.tool]` | Both LLM and tool spans |

This is powerful for agent scenarios where you want different scoring logic for different span types.

In [ ]:
from galileo.schema.metrics import StepType


def tool_output_json_scorer(span, **kwargs) -> float:
    """Score tool spans on whether their output is valid JSON."""
    import json
    try:
        json.loads(str(span.output or ''))
        return 1.0
    except (json.JSONDecodeError, TypeError):
        return 0.0


def agent_decision_scorer(span, **kwargs) -> float:
    """Score agent spans on whether they produced a non-empty decision."""
    output = str(span.output or '')
    if len(output) > 10:
        return 1.0
    return 0.0


enable_metrics(
    project_name=PROJECT_NAME,
    log_stream_name=LOG_STREAM_NAME,
    metrics=[
        LocalMetricConfig(
            name='tool_json_validity',
            scorer_fn=tool_output_json_scorer,
            scorable_types=[StepType.tool],
        ),
        LocalMetricConfig(
            name='agent_decision_quality',
            scorer_fn=agent_decision_scorer,
            scorable_types=[StepType.agent],
        ),
    ],
)
'Enabled span-level metrics: tool_json_validity (tool spans) + agent_decision_quality (agent spans)'

## 9. LangChain Integration (Placeholder)

Galileo supports integration with **LangChain** via a callback handler. When available, the `GalileoCallback` automatically instruments LangChain chains and agents — every LLM call, tool invocation, and chain step becomes a Galileo span without any manual logging.

### Theoretical Pattern

```python
# NOTE: GalileoCallback is not yet available in the current SDK version.
# This shows the expected pattern for when it ships.

# from galileo.langchain import GalileoCallback
# from langchain_openai import ChatOpenAI
# from langchain.agents import create_react_agent
#
# callback = GalileoCallback(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)
# llm = ChatOpenAI(model="gpt-4o-mini", callbacks=[callback])
#
# # Every LangChain call is now auto-logged to Galileo:
# agent = create_react_agent(llm, tools=[...], prompt=prompt)
# result = agent.invoke({"input": "What is the weather?"})
#
# # Galileo automatically creates:
# # Trace → AgentSpan → ToolSpan(s) → LLMSpan(s)
```

### How It Will Work

| LangChain Event | Galileo Span |
|-----------------|-------------|
| `on_llm_start` / `on_llm_end` | LLM span with input messages, output, tokens |
| `on_tool_start` / `on_tool_end` | Tool span with tool name, input, output |
| `on_chain_start` / `on_chain_end` | Workflow span grouping sub-steps |
| `on_agent_action` / `on_agent_finish` | Agent span with reasoning and decisions |

### Current Alternative

Until `GalileoCallback` ships, you can instrument LangChain manually using the patterns shown in this notebook:
- Use `@galileo_log` decorators on your tool functions (Step 3)
- Use `GalileoLogger` for manual trace construction (Step 4)
- Use distributed tracing headers (Step 7) if LangChain runs in a separate service

> **Status:** The LangChain callback handler is planned for a future SDK release. Check the [Galileo docs](https://v2docs.galileo.ai) for the latest integration status.

## 10. Clean Up the Demo Project

In [ ]:
delete_project(name=PROJECT_NAME)
